# Crear dataftames CSV

## Location

In [1]:
# Llegir el CSV conservant els codis com a text.
from pathlib import Path
import pandas as pd

data_dir = Path('dades')
if not data_dir.is_dir():
    data_dir = Path('VI_Project1/dades')

incendis_gencat = pd.read_csv(
    data_dir / 'Incendis_forestals_a_Catalunya._Anys_2011-2024_20261116.csv',
    dtype='string',
)
# ID propi segons la fila original del CSV (no es un ID oficial).
incendis_gencat['idincident'] = range(1, len(incendis_gencat) + 1)
incendis_gencat.head()

,DATA INCENDI,CODI COMARCA,COMARCA,CODI MUNICIPI,TERMEMUNIC,HAARBRADES,HANOARBRAD,HANOFOREST,HAFORESTAL,idincident
0,11/01/2011,08,Baix Camp,431299,Riudoms,"0,00","0,20","0,00","0,20",1
1,13/01/2011,37,Terra Alta,430483,Corbera d'Ebre,"0,20","0,00","0,00","0,20",2
2,14/01/2011,02,Alt Empordà,170545,Colera,"0,01","0,00","0,00","0,01",3
3,16/01/2011,06,Anoia,081022,Igualada,"0,00","0,02","0,00","0,02",4
4,16/01/2011,22,Montsià,430629,Freginals,"0,00","0,00","0,00","0,00",5


In [2]:
# Eliminar el digit de control per obtenir el codi municipal de cinc digits.
fires = incendis_gencat.copy()
municipality_ids = fires['CODI MUNICIPI'].str.strip()
assert municipality_ids.str.fullmatch(r'[0-9]{6}').fillna(False).all(), 'Review missing or malformed municipality IDs'
fires['codi_municipi'] = municipality_ids.str[:5]
fires[['CODI MUNICIPI', 'codi_municipi']].head()

,CODI MUNICIPI,codi_municipi
0,431299,43129
1,430483,43048
2,170545,17054
3,081022,08102
4,430629,43062


In [3]:
# Preparar els noms i les divisions territorials amb la taula local.
municipalities = pd.read_csv(
    data_dir / 'corrMpisComsAtps.csv', sep=';', skiprows=3, dtype='string',
    usecols=['Codi municipi', 'Nom municipi', 'Codi província', 'Nom província', 'Nom comarca'],
)
municipalities = municipalities.rename(columns={
    'Nom municipi': 'municipi', 'Codi província': 'idprovincia',
    'Nom província': 'provincia', 'Nom comarca': 'comarca',
})
municipalities['codi_municipi'] = municipalities['Codi municipi'].str.strip().str[:5]
municipalities = municipalities[['codi_municipi', 'idprovincia', 'municipi', 'provincia', 'comarca']]

# Llegir les coordenades municipals i convertir les comes decimals.
coords = pd.read_csv(data_dir / 'MUNICIPIOS.csv', sep=';', encoding='cp1252', dtype='string')
coords = coords.loc[coords['COD_PROV'].isin(['08', '17', '25', '43'])].copy()
coords['codi_municipi'] = coords['COD_INE'].str.strip().str[:5]
coords['latitud'] = pd.to_numeric(coords['LATITUD_ETRS89_REGCAN95'].str.replace(',', '.', regex=False), errors='coerce')
coords['longitud'] = pd.to_numeric(coords['LONGITUD_ETRS89_REGCAN95'].str.replace(',', '.', regex=False), errors='coerce')
coords = coords[['codi_municipi', 'latitud', 'longitud']]

In [4]:
# Unir per codi municipal sense perdre ni duplicar registres.
merged = (
    fires.merge(municipalities, on='codi_municipi', how='left', validate='many_to_one')
    .merge(coords, on='codi_municipi', how='left', validate='many_to_one')
)
assert len(merged) == len(incendis_gencat), 'The joins changed the number of fire records'
assert merged['idprovincia'].eq(merged['codi_municipi'].str[:2]).fillna(False).all(), 'Review unmatched or conflicting province IDs'

merged['idprovincia'] = merged['idprovincia'].astype('Int64')
merged['idmunicipio'] = merged['codi_municipi'].str[2:5].astype('Int64')
locations = merged[['idincident', 'idprovincia', 'idmunicipio', 'municipi', 'provincia', 'latitud', 'longitud', 'comarca']].copy()
locations.head()

,idincident,idprovincia,idmunicipio,municipi,provincia,latitud,longitud,comarca
0,1,43,129,Riudoms,Tarragona,41.137728,1.050838,Baix Camp
1,2,43,48,Corbera d'Ebre,Tarragona,41.077737,0.478135,Terra Alta
2,3,17,54,Colera,Girona,42.404019,3.150471,Alt Empordà
3,4,8,102,Igualada,Barcelona,41.578714,1.617676,Anoia
4,5,43,62,Freginals,Tarragona,40.671684,0.519778,Montsià


In [5]:
# Comprovar les files i els valors absents.
print(f'Source rows: {len(incendis_gencat):,} | Location rows: {len(locations):,}')
print('Missing values by column:')
print(locations.isna().sum())
locations.info()

Source rows: 7,545 | Location rows: 7,545
Missing values by column:
idincident     0
idprovincia    0
idmunicipio    0
municipi       0
provincia      0
latitud        0
longitud       0
comarca        0
dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 7545 entries, 0 to 7544
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   idincident   7545 non-null   int64  
 1   idprovincia  7545 non-null   Int64  
 2   idmunicipio  7545 non-null   Int64  
 3   municipi     7545 non-null   string 
 4   provincia    7545 non-null   string 
 5   latitud      7545 non-null   Float64
 6   longitud     7545 non-null   Float64
 7   comarca      7545 non-null   string 
dtypes: Float64(2), Int64(2), int64(1), string(3)
memory usage: 501.2 KB
